# Golbal Threshold tuning

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)


In [2]:
OUTPUT_DIR = Path(
    r"C:\Users\alrazz\Downloads\_AriaBERT_finetuned_evaluation\SP_ID_hybrid"
)


In [3]:
THRESHOLD_START = 0.01
THRESHOLD_END = 0.99
THRESHOLD_STEP = 0.01

In [4]:
MODEL_LABELS = [
    "MT",   # 0
    "LY",   # 1
    "SP",   # 2
    "ID",   # 3
    "NA",   # 4
    "HI",   # 5
    "IN",   # 6
    "OP",   # 7
    "IP",   # 8
    "it",   # 9
    "ne",   # 10
    "sr",   # 11
    "nb",   # 12
    "re",   # 13
    "en",   # 14
    "ra",   # 15
    "dtp",  # 16
    "fi",   # 17
    "lt",   # 18
    "rv",   # 19
    "ob",   # 20
    "rs",   # 21
    "av",   # 22
    "ds",   # 23
    "ed",   # 24
]


## Load Data

In [5]:
logits_file = OUTPUT_DIR / "dev_logits.npy"
labels_file = OUTPUT_DIR / "dev_labels.npy"

In [6]:
if not logits_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{logits_file}"
    )

if not labels_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{labels_file}"
    )


In [7]:
print("=" * 80)
print("GLOBAL THRESHOLD TUNING")
print("=" * 80)

print("\nLoading Dev logits...")
dev_logits = np.load(logits_file)

print("Loading Dev labels...")
dev_labels = np.load(labels_file)

print("\nLogits shape:", dev_logits.shape)
print("Labels shape:", dev_labels.shape)


if dev_logits.shape != dev_labels.shape:
    raise ValueError(
        "Logits and labels have different shapes:\n"
        f"logits: {dev_logits.shape}\n"
        f"labels: {dev_labels.shape}"
    )

GLOBAL THRESHOLD TUNING

Loading Dev logits...
Loading Dev labels...

Logits shape: (609, 25)
Labels shape: (609, 25)


## LABEL NAMES

In [8]:
num_classes = dev_labels.shape[1]

if MODEL_LABELS is None:

    model_labels = [
        f"class_{i}"
        for i in range(num_classes)
    ]

else:

    model_labels = MODEL_LABELS

    if len(model_labels) != num_classes:
        raise ValueError(
            f"Number of labels ({len(model_labels)}) "
            f"does not match number of classes ({num_classes})."
        )


print("\nNumber of classes:", num_classes)


Number of classes: 25


## LOGITS -> PROBABILITIES

In [9]:
print("\nConverting logits to probabilities...")

# Numerically stable sigmoid
dev_probabilities = np.empty_like(
    dev_logits,
    dtype=np.float64,
)

positive_mask = dev_logits >= 0

dev_probabilities[positive_mask] = (
    1.0
    / (
        1.0
        + np.exp(
            -dev_logits[positive_mask]
        )
    )
)

dev_probabilities[~positive_mask] = (
    np.exp(
        dev_logits[~positive_mask]
    )
    / (
        1.0
        + np.exp(
            dev_logits[~positive_mask]
        )
    )
)



Converting logits to probabilities...


## THRESHOLD SEARCH

In [10]:
thresholds = np.arange(
    THRESHOLD_START,
    THRESHOLD_END + THRESHOLD_STEP / 2,
    THRESHOLD_STEP,
)

results = []


print(
    f"\nTesting {len(thresholds)} thresholds..."
)

for threshold in thresholds:

    predictions = (
        dev_probabilities >= threshold
    ).astype(np.int32)


    macro_f1 = f1_score(
        dev_labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    micro_f1 = f1_score(
        dev_labels,
        predictions,
        average="micro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        dev_labels,
        predictions,
        average="weighted",
        zero_division=0,
    )

    macro_precision = precision_score(
        dev_labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    macro_recall = recall_score(
        dev_labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    micro_precision = precision_score(
        dev_labels,
        predictions,
        average="micro",
        zero_division=0,
    )

    micro_recall = recall_score(
        dev_labels,
        predictions,
        average="micro",
        zero_division=0,
    )

    results.append({
        "threshold": threshold,
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "weighted_f1": weighted_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
    })


results_df = pd.DataFrame(results)


Testing 99 thresholds...


## FIND BEST THRESHOLD

In [11]:
best_index = results_df[
    "macro_f1"
].idxmax()

best_row = results_df.loc[
    best_index
]

best_threshold = float(
    best_row["threshold"]
)

best_macro_f1 = float(
    best_row["macro_f1"]
)


print("\n" + "=" * 80)
print("BEST GLOBAL THRESHOLD")
print("=" * 80)

print(
    f"\nBest threshold: {best_threshold:.4f}"
)

print(
    f"Macro F1:       {best_row['macro_f1']:.6f}"
)

print(
    f"Micro F1:        {best_row['micro_f1']:.6f}"
)

print(
    f"Weighted F1:     {best_row['weighted_f1']:.6f}"
)

print(
    f"Macro Precision: {best_row['macro_precision']:.6f}"
)

print(
    f"Macro Recall:    {best_row['macro_recall']:.6f}"
)

print(
    f"Micro Precision: {best_row['micro_precision']:.6f}"
)

print(
    f"Micro Recall:    {best_row['micro_recall']:.6f}"
)



BEST GLOBAL THRESHOLD

Best threshold: 0.4000
Macro F1:       0.185284
Micro F1:        0.198471
Weighted F1:     0.301031
Macro Precision: 0.109199
Macro Recall:    0.973133
Micro Precision: 0.110363
Micro Recall:    0.984233


## SAVE ALL THRESHOLD RESULTS

In [12]:
results_file = (
    OUTPUT_DIR
    / "global_threshold_results.csv"
)

results_df.to_csv(
    results_file,
    index=False,
)

print(
    f"\nSaved threshold results to:\n"
    f"{results_file}"
)


Saved threshold results to:
C:\Users\alrazz\Downloads\_AriaBERT_finetuned_evaluation\SP_ID_hybrid\global_threshold_results.csv


## SAVE BEST THRESHOLD

In [13]:
best_threshold_file = (
    OUTPUT_DIR
    / "best_global_threshold.json"
)

best_threshold_data = {
    "threshold": best_threshold,
    "macro_f1": best_macro_f1,
    "micro_f1": float(
        best_row["micro_f1"]
    ),
    "weighted_f1": float(
        best_row["weighted_f1"]
    ),
    "macro_precision": float(
        best_row["macro_precision"]
    ),
    "macro_recall": float(
        best_row["macro_recall"]
    ),
    "micro_precision": float(
        best_row["micro_precision"]
    ),
    "micro_recall": float(
        best_row["micro_recall"]
    ),
}

with open(
    best_threshold_file,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        best_threshold_data,
        f,
        indent=2,
    )

print(
    f"Saved best threshold to:\n"
    f"{best_threshold_file}"
)



Saved best threshold to:
C:\Users\alrazz\Downloads\_AriaBERT_finetuned_evaluation\SP_ID_hybrid\best_global_threshold.json


## PER-CLASS PERFORMANCE AT BEST GLOBAL THRESHOLD

In [14]:
best_predictions = (
    dev_probabilities >= best_threshold
).astype(np.int32)


per_class_f1 = f1_score(
    dev_labels,
    best_predictions,
    average=None,
    zero_division=0,
)

per_class_precision = precision_score(
    dev_labels,
    best_predictions,
    average=None,
    zero_division=0,
)

per_class_recall = recall_score(
    dev_labels,
    best_predictions,
    average=None,
    zero_division=0,
)


# Number of actual positives
actual_positives = dev_labels.sum(
    axis=0
)

# Number of predicted positives
predicted_positives = best_predictions.sum(
    axis=0
)


per_class_df = pd.DataFrame({
    "label": model_labels,
    "f1": per_class_f1,
    "precision": per_class_precision,
    "recall": per_class_recall,
    "actual_positives": actual_positives,
    "predicted_positives": predicted_positives,
})


per_class_file = (
    OUTPUT_DIR
    / "global_threshold_per_class_f1.csv"
)

per_class_df.to_csv(
    per_class_file,
    index=False,
)

print(
    f"\nSaved per-class results to:\n"
    f"{per_class_file}"
)


Saved per-class results to:
C:\Users\alrazz\Downloads\_AriaBERT_finetuned_evaluation\SP_ID_hybrid\global_threshold_per_class_f1.csv


## PLOT

In [15]:
plt.figure(
    figsize=(10, 6)
)

plt.plot(
    results_df["threshold"],
    results_df["macro_f1"],
    label="Macro F1",
    linewidth=2,
)

plt.plot(
    results_df["threshold"],
    results_df["micro_f1"],
    label="Micro F1",
    linewidth=2,
)

plt.plot(
    results_df["threshold"],
    results_df["weighted_f1"],
    label="Weighted F1",
    linewidth=2,
)

plt.axvline(
    best_threshold,
    color="red",
    linestyle="--",
    linewidth=2,
    label=(
        f"Best threshold = "
        f"{best_threshold:.2f}"
    ),
)

plt.xlabel("Global Threshold")
plt.ylabel("F1")
plt.title(
    "Global Threshold Tuning on Dev Set"
)

plt.grid(
    True,
    alpha=0.3,
)

plt.legend()

plt.tight_layout()


plot_file = (
    OUTPUT_DIR
    / "global_threshold_curve.png"
)

plt.savefig(
    plot_file,
    dpi=300,
)

plt.close()

print(
    f"Saved plot to:\n"
    f"{plot_file}"
)


Saved plot to:
C:\Users\alrazz\Downloads\_AriaBERT_finetuned_evaluation\SP_ID_hybrid\global_threshold_curve.png


In [16]:
print("\n" + "=" * 80)
print("GLOBAL THRESHOLD TUNING COMPLETE")
print("=" * 80)


GLOBAL THRESHOLD TUNING COMPLETE


# Per-class Threshold Tuning

In [17]:
OUTPUT_DIR = Path(
    r"C:\Users\alrazz\Downloads\_AriaBERT_finetuned_evaluation\SP_ID_hybrid"
)

THRESHOLD_START = 0.01
THRESHOLD_END = 0.99
THRESHOLD_STEP = 0.01

In [18]:
MODEL_LABELS = [
    "MT",   # 0
    "LY",   # 1
    "SP",   # 2
    "ID",   # 3
    "NA",   # 4
    "HI",   # 5
    "IN",   # 6
    "OP",   # 7
    "IP",   # 8
    "it",   # 9
    "ne",   # 10
    "sr",   # 11
    "nb",   # 12
    "re",   # 13
    "en",   # 14
    "ra",   # 15
    "dtp",  # 16
    "fi",   # 17
    "lt",   # 18
    "rv",   # 19
    "ob",   # 20
    "rs",   # 21
    "av",   # 22
    "ds",   # 23
    "ed",   # 24
]


## Load data

In [19]:
logits_file = OUTPUT_DIR / "dev_logits.npy"
labels_file = OUTPUT_DIR / "dev_labels.npy"

if not logits_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{logits_file}"
    )

if not labels_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{labels_file}"
    )


print("=" * 80)
print("PER-CLASS THRESHOLD TUNING")
print("=" * 80)


print("\nLoading Dev logits...")
dev_logits = np.load(logits_file)

print("Loading Dev labels...")
dev_labels = np.load(labels_file)


print("\nLogits shape:", dev_logits.shape)
print("Labels shape:", dev_labels.shape)


if dev_logits.shape != dev_labels.shape:
    raise ValueError(
        "Logits and labels have different shapes:\n"
        f"logits: {dev_logits.shape}\n"
        f"labels: {dev_labels.shape}"
    )



PER-CLASS THRESHOLD TUNING

Loading Dev logits...
Loading Dev labels...

Logits shape: (609, 25)
Labels shape: (609, 25)


## Label names

In [20]:
num_classes = dev_labels.shape[1]

if MODEL_LABELS is None:

    model_labels = [
        f"class_{i}"
        for i in range(num_classes)
    ]

else:

    model_labels = MODEL_LABELS

    if len(model_labels) != num_classes:
        raise ValueError(
            f"Number of labels ({len(model_labels)}) "
            f"does not match number of classes ({num_classes})."
        )


## LOGITS -> PROBABILITIES

In [21]:
print("\nConverting logits to probabilities...")

dev_probabilities = np.empty_like(
    dev_logits,
    dtype=np.float64,
)

positive_mask = dev_logits >= 0

dev_probabilities[positive_mask] = (
    1.0
    / (
        1.0
        + np.exp(
            -dev_logits[positive_mask]
        )
    )
)

dev_probabilities[~positive_mask] = (
    np.exp(
        dev_logits[~positive_mask]
    )
    / (
        1.0
        + np.exp(
            dev_logits[~positive_mask]
        )
    )
)



Converting logits to probabilities...


## THRESHOLDS TO TEST

In [22]:
thresholds = np.arange(
    THRESHOLD_START,
    THRESHOLD_END + THRESHOLD_STEP / 2,
    THRESHOLD_STEP,
)

## FIND BEST THRESHOLD FOR EACH CLASS

In [23]:
optimal_thresholds = []

per_class_results = []


print("\n" + "=" * 80)
print("SEARCHING FOR OPTIMAL THRESHOLD FOR EACH CLASS")
print("=" * 80)


for class_idx in range(num_classes):

    label_name = model_labels[class_idx]

    y_true = dev_labels[
        :, class_idx
    ]

    y_prob = dev_probabilities[
        :, class_idx
    ]


    best_f1 = -1.0
    best_threshold = 0.5
    best_precision = 0.0
    best_recall = 0.0


    for threshold in thresholds:

        y_pred = (
            y_prob >= threshold
        ).astype(np.int32)


        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0,
        )

        precision = precision_score(
            y_true,
            y_pred,
            zero_division=0,
        )

        recall = recall_score(
            y_true,
            y_pred,
            zero_division=0,
        )


        if f1 > best_f1:

            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall


    actual_positives = int(
        y_true.sum()
    )

    predictions_at_best = (
        y_prob >= best_threshold
    ).astype(np.int32)

    predicted_positives = int(
        predictions_at_best.sum()
    )


    optimal_thresholds.append(
        best_threshold
    )


    per_class_results.append({

        "class_index": class_idx,

        "label": label_name,

        "optimal_threshold": best_threshold,

        "f1": best_f1,

        "precision": best_precision,

        "recall": best_recall,

        "actual_positives": actual_positives,

        "predicted_positives": predicted_positives,

    })


    print(
        f"{class_idx:3d} | "
        f"{label_name:30s} | "
        f"threshold={best_threshold:.2f} | "
        f"F1={best_f1:.4f} | "
        f"P={best_precision:.4f} | "
        f"R={best_recall:.4f}"
    )


per_class_df = pd.DataFrame(
    per_class_results
)




SEARCHING FOR OPTIMAL THRESHOLD FOR EACH CLASS
  0 | MT                             | threshold=0.60 | F1=0.2609 | P=0.1682 | R=0.5806
  1 | LY                             | threshold=0.01 | F1=0.1766 | P=0.0969 | R=1.0000
  2 | SP                             | threshold=0.46 | F1=0.5687 | P=0.4124 | R=0.9159
  3 | ID                             | threshold=0.46 | F1=0.3343 | P=0.2017 | R=0.9754
  4 | NA                             | threshold=0.55 | F1=0.2113 | P=0.1596 | R=0.3125
  5 | HI                             | threshold=0.48 | F1=0.1250 | P=0.0775 | R=0.3226
  6 | IN                             | threshold=0.43 | F1=0.5051 | P=0.3413 | R=0.9707
  7 | OP                             | threshold=0.01 | F1=0.4926 | P=0.3268 | R=1.0000
  8 | IP                             | threshold=0.47 | F1=0.1824 | P=0.1003 | R=1.0000
  9 | it                             | threshold=0.44 | F1=0.1488 | P=0.0856 | R=0.5682
 10 | ne                             | threshold=0.57 | F1=0.1843 | P=0.

## SAVE PER-CLASS THRESHOLDS

In [24]:
threshold_file = (
    OUTPUT_DIR
    / "per_class_thresholds.csv"
)

per_class_df.to_csv(
    threshold_file,
    index=False,
)


print(
    f"\nSaved per-class thresholds to:\n"
    f"{threshold_file}"
)


# JSON version

threshold_dict = {
    model_labels[i]: float(
        optimal_thresholds[i]
    )
    for i in range(num_classes)
}


threshold_json_file = (
    OUTPUT_DIR
    / "per_class_thresholds.json"
)

with open(
    threshold_json_file,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        threshold_dict,
        f,
        indent=2,
    )


print(
    f"Saved JSON thresholds to:\n"
    f"{threshold_json_file}"
)


Saved per-class thresholds to:
C:\Users\alrazz\Downloads\_AriaBERT_finetuned_evaluation\SP_ID_hybrid\per_class_thresholds.csv
Saved JSON thresholds to:
C:\Users\alrazz\Downloads\_AriaBERT_finetuned_evaluation\SP_ID_hybrid\per_class_thresholds.json


## APPLY PER-CLASS THRESHOLDS

In [25]:

optimal_thresholds_array = np.asarray(
    optimal_thresholds
)


per_class_predictions = (
    dev_probabilities
    >= optimal_thresholds_array
).astype(np.int32)


## OVERALL PERFORMANCE

In [26]:
macro_f1 = f1_score(
    dev_labels,
    per_class_predictions,
    average="macro",
    zero_division=0,
)

micro_f1 = f1_score(
    dev_labels,
    per_class_predictions,
    average="micro",
    zero_division=0,
)

weighted_f1 = f1_score(
    dev_labels,
    per_class_predictions,
    average="weighted",
    zero_division=0,
)

macro_precision = precision_score(
    dev_labels,
    per_class_predictions,
    average="macro",
    zero_division=0,
)

macro_recall = recall_score(
    dev_labels,
    per_class_predictions,
    average="macro",
    zero_division=0,
)

micro_precision = precision_score(
    dev_labels,
    per_class_predictions,
    average="micro",
    zero_division=0,
)

micro_recall = recall_score(
    dev_labels,
    per_class_predictions,
    average="micro",
    zero_division=0,
)


In [27]:
print("\n" + "=" * 80)
print("PER-CLASS THRESHOLD RESULTS")
print("=" * 80)

print(
    f"\nMacro F1:       {macro_f1:.6f}"
)

print(
    f"Micro F1:        {micro_f1:.6f}"
)

print(
    f"Weighted F1:     {weighted_f1:.6f}"
)

print(
    f"Macro Precision: {macro_precision:.6f}"
)

print(
    f"Macro Recall:    {macro_recall:.6f}"
)

print(
    f"Micro Precision: {micro_precision:.6f}"
)

print(
    f"Micro Recall:    {micro_recall:.6f}"
)



PER-CLASS THRESHOLD RESULTS

Macro F1:       0.201886
Micro F1:        0.227959
Weighted F1:     0.316547
Macro Precision: 0.125019
Macro Recall:    0.767960
Micro Precision: 0.131921
Micro Recall:    0.838084


## SAVE SUMMARY

In [28]:

summary_df = pd.DataFrame([{

    "method": "per_class_thresholds",

    "macro_f1": macro_f1,

    "micro_f1": micro_f1,

    "weighted_f1": weighted_f1,

    "macro_precision": macro_precision,

    "macro_recall": macro_recall,

    "micro_precision": micro_precision,

    "micro_recall": micro_recall,

}])


summary_file = (
    OUTPUT_DIR
    / "per_class_threshold_summary.csv"
)

summary_df.to_csv(
    summary_file,
    index=False,
)


print(
    f"\nSaved summary to:\n"
    f"{summary_file}"
)




Saved summary to:
C:\Users\alrazz\Downloads\_AriaBERT_finetuned_evaluation\SP_ID_hybrid\per_class_threshold_summary.csv


## F1 CURVES FOR EACH CLASS

In [29]:
fig, ax = plt.subplots(
    figsize=(16, 9)
)

for class_idx in range(num_classes):

    y_true = dev_labels[:, class_idx]

    y_prob = dev_probabilities[:, class_idx]

    class_f1_scores = []

    for threshold in thresholds:

        y_pred = (
            y_prob >= threshold
        ).astype(np.int32)

        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0,
        )

        class_f1_scores.append(f1)

    ax.plot(
        thresholds,
        class_f1_scores,
        linewidth=1.5,
        alpha=0.7,
        label=model_labels[class_idx],
    )


ax.set_xlabel(
    "Threshold"
)

ax.set_ylabel(
    "F1"
)

ax.set_title(
    "Per-Class F1 vs Threshold"
)

ax.grid(
    True,
    alpha=0.3,
)

# Legend outside the plot
ax.legend(
    title="Class",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    fontsize=9,
    title_fontsize=10,
    frameon=True,
)

# Leave enough space for the external legend
plt.tight_layout(
    rect=[0, 0, 0.82, 1]
)


plot_file = (
    OUTPUT_DIR
    / "per_class_threshold_f1_curve.png"
)

plt.savefig(
    plot_file,
    dpi=300,
    bbox_inches="tight",
)

plt.close()


print(
    f"Saved F1 curves to:\n"
    f"{plot_file}"
)

Saved F1 curves to:
C:\Users\alrazz\Downloads\_AriaBERT_finetuned_evaluation\SP_ID_hybrid\per_class_threshold_f1_curve.png


In [30]:
print("\n" + "=" * 80)
print("PER-CLASS THRESHOLD TUNING COMPLETE")
print("=" * 80)


PER-CLASS THRESHOLD TUNING COMPLETE


# Checking on test set

In [31]:
# ============================================================
# LOAD TEST DATA
# ============================================================

test_logits_file = OUTPUT_DIR / "test_logits.npy"
test_labels_file = OUTPUT_DIR / "test_labels.npy"

if not test_logits_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{test_logits_file}"
    )

if not test_labels_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{test_labels_file}"
    )

test_logits = np.load(test_logits_file)
test_labels = np.load(test_labels_file)

print("\nTest logits shape:", test_logits.shape)
print("Test labels shape:", test_labels.shape)

if test_logits.shape != test_labels.shape:
    raise ValueError(
        "Test logits and labels have different shapes."
    )



Test logits shape: (633, 25)
Test labels shape: (633, 25)


In [32]:
#convert test logits
test_probabilities = np.empty_like(
    test_logits,
    dtype=np.float64,
)

positive_mask = test_logits >= 0

test_probabilities[positive_mask] = (
    1.0 /
    (
        1.0 +
        np.exp(-test_logits[positive_mask])
    )
)

test_probabilities[~positive_mask] = (
    np.exp(test_logits[~positive_mask])
    /
    (
        1.0 +
        np.exp(test_logits[~positive_mask])
    )
)


In [33]:
test_pred_05 = (
    test_probabilities >= 0.50
).astype(np.int32)

test_pred_global = (
    test_probabilities >= best_threshold
).astype(np.int32)

test_pred_per_class = (
    test_probabilities >= optimal_thresholds_array
).astype(np.int32)


In [34]:
def calculate_metrics(y_true, y_pred):

    return {
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "micro_f1": f1_score(
            y_true,
            y_pred,
            average="micro",
            zero_division=0,
        ),

        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),

        "macro_precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "macro_recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "micro_precision": precision_score(
            y_true,
            y_pred,
            average="micro",
            zero_division=0,
        ),

        "micro_recall": recall_score(
            y_true,
            y_pred,
            average="micro",
            zero_division=0,
        ),
    }


In [35]:
test_results = pd.DataFrame([
    {
        "method": "0.50",
        "threshold": 0.50,
        **calculate_metrics(
            test_labels,
            test_pred_05,
        ),
    },
    {
        "method": "global",
        "threshold": best_threshold,
        **calculate_metrics(
            test_labels,
            test_pred_global,
        ),
    },
    {
        "method": "per_class",
        "threshold": "individual",
        **calculate_metrics(
            test_labels,
            test_pred_per_class,
        ),
    },
])


In [37]:
test_results

,method,threshold,macro_f1,micro_f1,weighted_f1,macro_precision,macro_recall,micro_precision,micro_recall
0,0.50,0.5,0.124526,0.162319,0.182347,0.095317,0.469760,0.098962,0.451168
1,global,0.01,0.180230,0.190808,0.291109,0.105466,1.000000,0.105466,1.000000
2,per_class,individual,0.193855,0.216408,0.301229,0.119366,0.753471,0.124647,0.820252


In [38]:
test_results_file = (
    OUTPUT_DIR / "test_threshold_comparison.csv"
)

test_results.to_csv(
    test_results_file,
    index=False,
)

print(
    f"\nSaved test results to:\n"
    f"{test_results_file}"
)



Saved test results to:
C:\Users\alrazz\Downloads\_AriaBERT_finetuned_evaluation\SP_ID_hybrid\test_threshold_comparison.csv
